# 03 | Foreign Key and Table Relationship Analysis

**Purpose:** Check whether key values in child tables match their referenced parent tables, and inspect geographic-prefix and product-category mapping gaps.

**Inputs:** The nine original Olist CSV files in `data/raw/`.

**Interpretation:** Unmatched values are data-quality or mapping findings to investigate; they are not automatically a reason to delete records. Geolocation zip-prefix checks are membership tests against the geolocation table, which can contain repeated prefixes.

Run after `01_dataset_audit.ipynb` and `02_primary_key_identifier_analysis.ipynb`.


In [ ]:
import pandas as pd
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = Path(os.environ.get("OLIST_DATA_DIR", PROJECT_ROOT / "data" / "raw")).expanduser()

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Olist CSV folder not found: {DATA_DIR}\n"
        "Download the Olist CSV files and place them in data/raw, "
        "or set the OLIST_DATA_DIR environment variable."
    )

print("Dataset folder:", DATA_DIR)


## Load the source tables


In [ ]:
required_csv_files = [
    "olist_orders_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv",
]
available_csvs = {file.name: file for file in DATA_DIR.glob("*.csv")}
missing_files = sorted(set(required_csv_files) - set(available_csvs))
if missing_files:
    raise FileNotFoundError("Missing required Olist CSV files: " + ", ".join(missing_files))

datasets = {
    Path(file_name).stem: pd.read_csv(available_csvs[file_name], low_memory=False)
    for file_name in required_csv_files
}

# Normalize an optional BOM/malformed BOM prefix on the category translation key.
translation_table = datasets["product_category_name_translation"]
translation_category_column = next(
    (column for column in translation_table.columns
     if str(column).replace("\ufeff", "").replace("ï»¿", "") == "product_category_name"),
    None,
)
if translation_category_column is None:
    raise KeyError("Could not find product_category_name in the category translation table")
if translation_category_column != "product_category_name":
    translation_table.rename(columns={translation_category_column: "product_category_name"}, inplace=True)

print("Datasets loaded:", len(datasets))

## Validate key relationships between tables


In [ ]:
relationships = [
    ("olist_orders_dataset", "customer_id", "olist_customers_dataset", "customer_id"),
    ("olist_order_items_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_order_items_dataset", "product_id", "olist_products_dataset", "product_id"),
    ("olist_order_items_dataset", "seller_id", "olist_sellers_dataset", "seller_id"),
    ("olist_order_payments_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_order_reviews_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_products_dataset", "product_category_name", "product_category_name_translation", "product_category_name"),
]

relationship_results = []
for child, child_column, parent, parent_column in relationships:
    child_df = datasets[child]
    parent_values = datasets[parent][parent_column].dropna()
    missing_count = int(child_df[child_column].isna().sum())
    unmatched_mask = child_df[child_column].notna() & ~child_df[child_column].isin(parent_values)
    relationship_results.append({
        "Child Table": child,
        "Parent Table": parent,
        "Child Key": child_column,
        "Parent Key": parent_column,
        "Child Rows": len(child_df),
        "Missing Child Keys": missing_count,
        "Non-null Unmatched Rows": int(unmatched_mask.sum()),
        "Distinct Unmatched Values": int(child_df.loc[unmatched_mask, child_column].nunique()),
    })

relationship_df = pd.DataFrame(relationship_results)
display(relationship_df)

## Check customer and seller zip-prefix coverage

Null or blank prefixes are counted separately from non-null prefixes that do not appear in the geolocation reference. Prefixes are normalized to five-character strings for the membership check.

In [ ]:
geo = datasets["olist_geolocation_dataset"]
geo_prefix_column = "geolocation_zip_code_prefix"

def normalize_zip_prefix(values):
    numeric_values = pd.to_numeric(values, errors="coerce").astype("Int64")
    return numeric_values.astype("string").str.zfill(5)

geo_prefixes = set(normalize_zip_prefix(geo[geo_prefix_column]).dropna())
zip_results = []

for table, column in [
    ("olist_customers_dataset", "customer_zip_code_prefix"),
    ("olist_sellers_dataset", "seller_zip_code_prefix"),
]:
    df = datasets[table]
    raw_prefixes = df[column].astype("string").str.strip()
    missing_mask = raw_prefixes.isna() | raw_prefixes.eq("")
    normalized_prefixes = normalize_zip_prefix(df[column])
    unmatched_mask = ~missing_mask & ~normalized_prefixes.isin(geo_prefixes)

    zip_results.append({
        "Dataset": table,
        "Total Rows": len(df),
        "Missing Prefix Rows": int(missing_mask.sum()),
        "Non-null Unmatched Rows": int(unmatched_mask.sum()),
        "Distinct Unmatched Prefixes": int(normalized_prefixes[unmatched_mask].nunique()),
    })

zip_coverage_df = pd.DataFrame(zip_results)
display(zip_coverage_df)

## Check product-category translation coverage

Missing product categories are counted separately from non-null categories that do not have a matching translation.

In [ ]:
products = datasets["olist_products_dataset"]
translation = datasets["product_category_name_translation"]
translated_categories = set(
    translation["product_category_name"].dropna().astype("string").str.strip()
)
product_categories = products["product_category_name"].astype("string").str.strip()
missing_category_mask = product_categories.isna() | product_categories.eq("")
untranslated_mask = ~missing_category_mask & ~product_categories.isin(translated_categories)

category_coverage = pd.DataFrame([{
    "Product Rows": len(products),
    "Missing Category Rows": int(missing_category_mask.sum()),
    "Non-null Untranslated Rows": int(untranslated_mask.sum()),
    "Distinct Untranslated Categories": int(product_categories[untranslated_mask].nunique()),
}])

display(category_coverage)
display(product_categories[untranslated_mask].value_counts().rename("Product Rows"))